## Notebook Flow

This notebook flattens raw JSON files from Bronze Landing into a 
structured Delta table in Bronze Ingestion. The pipeline executes 
in five sequential steps:

1. **Imports & configuration loading**  
   Load the metadata-driven schema mapping from `config/alpha_vantage_schema.json`.

2. **Define `parse_file()`**  
   Generic function to flatten a single JSON file into a list of 
   row dicts, using the schema config.

3. **List source files and apply the parser**  
   Iterate over all JSON files in Bronze Landing and accumulate 
   the parsed rows in memory (`all_rows`).

4. **Convert in-memory rows to a Spark DataFrame**  
   Transform the Python list of dicts into a distributed Spark 
   DataFrame for further processing.

5. **Write to ADLS as a Delta table — Bronze Ingestion**  
   Persist the DataFrame as a Delta table using overwrite semantics, 
   producing a clean snapshot ready for the Silver layer.

## Strategy: Overwrite vs Append

This notebook flattens all JSON files from Bronze Landing into a 
single Delta table in Bronze Ingestion using **OVERWRITE** semantics.

### Why overwrite?
- Bronze Ingestion is a **snapshot** of all valid records currently 
  in Bronze Landing
- Each run reflects the current state of source files
- No deduplication logic needed at this layer
- Bronze Landing remains immutable as the source of truth

### Where deduplication happens
- The Silver layer applies `MERGE` / `dropDuplicates` on the 
  natural key (`symbol`, `trade_date`)
- This ensures historical fact records remain consistent even when 
  the same data is reprocessed multiple times

### SCD considerations
- `fact_prices` does not require SCD — prices are immutable historical facts
- `dim_stock` could use SCD Type 1 (overwrite) for `last_refreshed` 
  and `time_zone`, since these reflect the current state

## Refactoring: From Hardcoded to Metadata-Driven

The initial implementation extracted JSON fields using hardcoded keys.
This works but couples the function to the API schema — any change in 
the source structure requires modifying the function logic.

To make this pipeline production-ready, the field mappings are now 
externalised to a configuration file (`config/alpha_vantage_schema.json`).

Benefits:
- Schema changes do not require code changes
- The function becomes generic and reusable across similar APIs
- Configuration is versioned independently from logic
- New fields can be added by editing the config file alone

This pattern is known as a Metadata-Driven Pipeline.

In [0]:
# import json


# path = "abfss://bronze@marketpulsedatalake.dfs.core.windows.net/stocks/ingest_date=2026-04-29/AAPL_20260429_115810.json"
# def parse_file(path: str) -> list[dict]:
#     """
#     Reads a raw Alpha Vantage JSON file from Bronze Landing and 
#     flattens its nested structure into a list of flat row dicts.
    
#     The Alpha Vantage TIME_SERIES_DAILY endpoint returns a nested 
#     JSON with two top-level keys ("Meta Data" and "Time Series (Daily)")
#     where each trading day is a dynamic key. This function extracts 
#     one flat row per trading day, preserving all values as strings 
#     (type casting is deferred to the Silver layer).
    
#     Args:
#         path: Full ADLS path to a single JSON file in Bronze Landing.
#               Example: "abfss://bronze@<account>.dfs.core.windows.net/
#                         stocks/ingest_date=YYYY-MM-DD/SYMBOL_TS.json"
    
#     Returns:
#         A list of dicts, one per trading day, with the following keys:
#             - symbol           (str): Stock ticker (e.g. "AAPL")
#             - trade_date       (str): Trading day in YYYY-MM-DD format
#             - last_refreshed   (str): Last refresh date from API metadata
#             - time_zone        (str): Stock exchange timezone
#             - open, high, low, close, volume (str): OHLCV values as strings
    
#     Example:
#         >>> rows = parse_file("abfss://.../AAPL_20260429.json")
#         >>> len(rows)
#         100
#         >>> rows[0]
#         {'symbol': 'AAPL', 'trade_date': '2026-04-28', 'open': '272.33', ...}
#     """
#     # Passo 1: ler o ficheiro JSON do ADLS
#     raw = dbutils.fs.head(path)
    
#     # Passo 2: parsear o JSON em dict Python
#     parsed = json.loads(raw)
    
#     # Passo 3: extrair as 3 informações do "Meta Data"
#     #          (symbol, last_refreshed, time_zone)
#     symbol = parsed["Meta Data"]["2. Symbol"]
#     last_refreshed = parsed["Meta Data"]["3. Last Refreshed"]
#     time_zone = parsed["Meta Data"]["5. Time Zone"]
    
#     # Passo 4: extrair o "Time Series (Daily)"
#     time_series = parsed["Time Series (Daily)"]
#     # Passo 5: criar uma lista vazia para guardar as linhas
#     parsed_rows = []
#     # Passo 6: para cada data no Time Series:
#     #          criar um dict com todas as colunas
#     #          adicionar à lista
#     for date, values in time_series.items():
#         parsed_rows.append({
#             "symbol": symbol,
#             "trade_date": date,
#             "last_refreshed": last_refreshed,
#             "time_zone": time_zone,
#             "open": values["1. open"],
#             "high": values["2. high"],
#             "low": values["3. low"],
#             "close": values["4. close"],
#             "volume": values["5. volume"]
#         })
    
#     # Passo 7: devolver a lista
#     return parsed_rows

    


In [0]:
# # Test the function with one file
# test_path = "abfss://bronze@marketpulsedatalake.dfs.core.windows.net/stocks/ingest_date=2026-04-29/AAPL_20260429_115810.json"
# result = parse_file(test_path)

# print(f"Number of rows: {len(result)}")
# print(f"First row: {result[0]}")

## Refactoring: From Hardcoded to Metadata-Driven

The initial implementation extracted JSON fields using hardcoded keys.
This works but couples the function to the API schema — any change in 
the source structure requires modifying the function logic.

To make this pipeline production-ready, the field mappings are now 
externalised to a configuration file (`config/alpha_vantage_schema.json`).

Benefits:
- Schema changes do not require code changes
- The function becomes generic and reusable across similar APIs
- Configuration is versioned independently from logic
- New fields can be added by editing the config file alone

This pattern is known as a Metadata-Driven Pipeline.

In [0]:
import json
import logging

with open("/Workspace/Repos/martalimas@gmail.com/market-pulse-pipeline/config/alpha_vantage_schema.json", "r") as f:
    schema_config = json.load(f)

META_FIELDS = schema_config["meta_fields"]
PRICE_FIELDS = schema_config["price_fields"]
print("META_FIELDS:", META_FIELDS)
print("PRICE_FIELDS:", PRICE_FIELDS)




In [0]:
def parse_file(path: str) -> list[dict]:
    '''read the description of the parse_file function above in cell 2'''
    raw = dbutils.fs.head(path)
    parsed = json.loads(raw)

    sections = schema_config["sections"]

    # Defensive check — required sections exist
    if not all(key in parsed for key in sections.values()):
        print(f"⚠️ Skipping file: {path}")
        return []
    
    meta_data = parsed[sections["meta"]]
    time_series = parsed[sections["time_series"]]
    
    # Extract metadata using the mapping
    
    meta_row = {
        new_name: meta_data[json_key] 
        for json_key, new_name in META_FIELDS.items()
    }
    
    # Iterate over each trading day
    
    parsed_rows = []
    
    for date, values in time_series.items():
        # Build a row by combining metadata + trade_date + price fields
        row = {
            **meta_row,                    # symbol, last_refreshed, time_zone
            "trade_date": date,
            **{
                new_name: values[json_key]
                for json_key, new_name in PRICE_FIELDS.items()
            }
        }
        parsed_rows.append(row)
    
    return parsed_rows

In [0]:
test_path = "abfss://bronze@marketpulsedatalake.dfs.core.windows.net/stocks/ingest_date=2026-04-29/AAPL_20260429_115810.json"
result = parse_file(test_path)
print(f"Number of rows: {len(result)}")

In [0]:
# Total de lines
print(f"Total rows: {len(result)}")

# first 3 lines
for row in result[:3]:
    print(row)

# last line
print("\nLast row:", result[-1])

In [0]:
#see it as a table in pandas
import pandas as pd
df_preview = pd.DataFrame(result)
df_preview.head(10)

In [0]:
landing_path = "abfss://bronze@marketpulsedatalake.dfs.core.windows.net/stocks/ingest_date=2026-04-29/"

# file listing
files = dbutils.fs.ls(landing_path)

# apply parse each file
all_rows = []
for f in files:
    all_rows.extend(parse_file(f.path))
    

print(f"Total rows from {len(files)} files: {len(all_rows)}")

In [0]:
#Write to ADLS as a Delta table
df = spark.createDataFrame(all_rows)
df.show(5)
df.printSchema()

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://bronze@marketpulsedatalake.dfs.core.windows.net/ingestion/stocks/") 

In [0]:
%sql

SELECT *
FROM delta.`abfss://bronze@marketpulsedatalake.dfs.core.windows.net/ingestion/stocks/`
WHERE symbol = 'AAPL'
LIMIT 10